# Correction models: навчання й незалежний тест
Достатньо CPU runtime. Ваги CAMP/RoMa не завантажуються: використовуємо вже збережені описи карти. Кожна модель повертає вісім імовірностей та емпіричний прогноз похибки.

In [ ]:
import os, sys
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/PhD/proj')
EXPERIMENT = PROJECT_ROOT / 'scripts/correction_models'
assert EXPERIMENT.is_dir(), f'Upload scripts/correction_models and scripts/visibility_localization: {EXPERIMENT}'
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
from scripts.correction_models.environment import configure_cuda_determinism
configure_cuda_determinism()


In [ ]:
REQUIREMENTS = 'requirements-cpu.txt'
import subprocess
from scripts.correction_models.environment import loaded_snapshot, require_clean_kernel, verify_scipy_imports
INSTALL_DEPENDENCIES = True  # After a requested session restart: False.
if INSTALL_DEPENDENCIES:
    before = loaded_snapshot()
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(EXPERIMENT / REQUIREMENTS)])
    require_clean_kernel(before)
verify_scipy_imports()


In [ ]:
import json
DATASET = EXPERIMENT / 'results/main-v3-cusolver/dataset.json'
RUN = EXPERIMENT / 'results/models-v3-cusolver'
TEST_OUTPUT = EXPERIMENT / 'results/test-v3-cusolver'
assert DATASET.is_file(), 'Complete the GPU collection first: dataset.json is published only after 1920 attempts.'
from scripts.correction_models.dataset import load_dataset, usable
data = load_dataset(DATASET)
print({s: len(usable(data['rows'], s)) for s in ('train', 'validation', 'calibration', 'test')})


## Навчання
Моделі навчаються лише на train. Температура калібрування — лише на calibration. Вибір кандидата — за validation log loss. Середня похибка не підміняє розподіл. Якщо певного класу не було в train, його згладжена ймовірність позначається як unresolved mass: для неї немає вигаданого вектора похибки.

In [ ]:
from scripts.correction_models.training import run_training
from scripts.correction_models.artifacts import read, load_verified
if not RUN.exists():
    run_training(DATASET, RUN)
development = read(RUN / 'summary.json')
print('Selected:', development['selected'])
print(json.dumps(development['classification'][development['selected']], indent=2))
print(json.dumps(development['distribution'][development['selected']], indent=2))


In [ ]:
from IPython.display import display, Image, Markdown
display(Markdown((RUN / 'report.md').read_text()))
display(Image(filename=str(RUN / 'calibration.png')))
display(Image(filename=str(RUN / 'territories.png')))
display(Image(filename=str(RUN / 'localizer_visibility.png')))


## Остаточний тест
Після фіксації моделей увімкніть RUN_FINAL_TEST. Вибраний кандидат уже зафіксований. Тест не використовуємо для нових налаштувань. Повторна комірка читає готовий звіт без повторного оцінювання. Чотири території — обмежена перевірка узагальнення; bootstrap-інтервали позначені exploratory.

In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    from scripts.correction_models.training import evaluate_frozen
    if not (TEST_OUTPUT / 'summary.json').exists():
        evaluate_frozen(RUN, DATASET, TEST_OUTPUT)
    final = read(TEST_OUTPUT / 'summary.json')
    print('Frozen selected:', final['selected'])
    print(json.dumps(final['classification'][final['selected']], indent=2))
    print(json.dumps(final['distribution'][final['selected']], indent=2))
    display(Image(filename=str(TEST_OUTPUT / 'calibration.png')))
    display(Image(filename=str(TEST_OUTPUT / 'territories.png')))
    display(Image(filename=str(TEST_OUTPUT / 'localizer_visibility.png')))


## Приклад виклику для планування
Використовуємо тільки збережений GT-free запит навчального кадру та карту. Еталонні координати, реальний кадр і метрики зіставлення не передаються моделі. Якщо обрано CAMP-кандидат, точний опис цього фрагмента читається з кешу без GPU. Для нового фрагмента потрібно обчислити опис незміненою CAMP; для нової карти — також перевірити домен.

In [ ]:
from scripts.correction_models.contracts import CorrectionQuery
from scripts.correction_models.prediction import load_predictor
from scripts.correction_models.embeddings import CachedCampEncoder
from scripts.visibility_localization.data import load_reference
selection = load_verified(EXPERIMENT / 'data/main-v1/selection.json')
row = next(r for r in data['rows'] if r['split'] == 'train' and r['features'] is not None)
source = next(s for s in selection['rows'] if s['scene_id'] == row['scene_id'])
reference = load_reference(source['reference_json'], 'aerial')
encoder = CachedCampEncoder(DATASET.parent / 'embeddings', data['provenance']['encoder'])
predictor = load_predictor(RUN, reference, encoder=encoder)
answer = predictor.predict_correction(CorrectionQuery(**row['query']))
print({k: v for k, v in answer.items() if k != 'accepted_error_distribution'})


Моделі збережені в JSON у `models/`, вибір — у `frozen_run.json`. Результат цього етапу не є підтвердженням ймовірності доставки чи швидкості на Jetson. Планувальник і маршрутний експеримент не запускаються.